# NHAMCS — Regional Mapping of Wait Time by Sex

**Goal:** colour a US map by Census region, showing median ED wait time for women, men, and the female-minus-male gap.

**Why region, not state:** NHAMCS public-use files suppress state for confidentiality. The geographic variable available is `REGION` (1=Northeast, 2=Midwest, 3=South, 4=West). Each US state on the map is therefore coloured by the value for its Census region.

**Data scope:** 2009–2022 (years where `WAITTIME` and triage acuity are reliably populated).

In [1]:
import io
import zipfile
from pathlib import Path

import certifi
import urllib3
import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = Path("../data/nhamcs")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE = (
    "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NHAMCS/stata"
)
HTTP = urllib3.PoolManager(ca_certs=certifi.where())

## 1. Load NHAMCS years 2009–2022

Same download routine as `01_explore.ipynb`, but here we only need years where `WAITTIME` and `IMMEDR` are both reliable. We also pull `REGION` this time.

In [2]:
YEARS = list(range(2009, 2023))
CORE = ["SEX", "AGE", "WAITTIME", "IMMEDR", "PAINSCALE", "REGION", "PATWT", "YEAR"]


def url_for(year: int) -> str:
    if year in (2006, 2007, 2008, 2009, 2010):
        return f"{BASE}/ed{year}-stata.exe"
    return f"{BASE}/ed{year}-stata.zip"


def dta_path(year: int) -> Path:
    return DATA_DIR / f"ed{year}-stata.dta"


def ensure_year(year: int) -> Path:
    target = dta_path(year)
    if target.exists():
        return target
    url = url_for(year)
    print(f"  fetching {year}: {url.split('/')[-1]}", end=" ")
    resp = HTTP.request("GET", url, preload_content=True)
    if resp.status != 200:
        raise RuntimeError(f"HTTP {resp.status}")
    with zipfile.ZipFile(io.BytesIO(resp.data)) as zf:
        dta_name = next((n for n in zf.namelist() if n.lower().endswith(".dta")), None)
        if dta_name is None:
            raise RuntimeError(f"No .dta file in archive for {year}")
        with zf.open(dta_name) as src, open(target, "wb") as dst:
            dst.write(src.read())
    print(f"-> {target.stat().st_size / 1e6:.1f} MB")
    return target


def load_year(year: int) -> pd.DataFrame:
    df = pd.read_stata(dta_path(year), convert_categoricals=False)
    df.columns = [c.upper() for c in df.columns]
    keep = [c for c in CORE if c in df.columns]
    out = df[keep].copy()
    out["YEAR"] = year
    return out


frames = []
for y in YEARS:
    try:
        ensure_year(y)
        frames.append(load_year(y))
    except Exception as e:
        print(f"  {y}: failed \u2014 {e}")

data = pd.concat(frames, ignore_index=True, sort=False)
for col in ["WAITTIME", "IMMEDR", "PAINSCALE", "REGION"]:
    if col in data.columns:
        data.loc[data[col] < 0, col] = np.nan
data["sex_label"] = data["SEX"].map({1: "Female", 2: "Male"})

print(f"Loaded {len(data):,} visits across {data['YEAR'].nunique()} years.")
print(f"Years: {sorted(data['YEAR'].unique())}")
print(f"Share with valid WAITTIME: {data['WAITTIME'].notna().mean():.1%}")
print(f"Share with valid REGION:   {data['REGION'].notna().mean():.1%}")

Loaded 323,137 visits across 14 years.
Years: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Share with valid WAITTIME: 85.8%
Share with valid REGION:   100.0%


## 2. Compute median wait time by region × sex

`REGION` codes (NHAMCS): 1 = Northeast, 2 = Midwest, 3 = South, 4 = West.

In [3]:
REGION_NAMES = {1: "Northeast", 2: "Midwest", 3: "South", 4: "West"}

valid = data.dropna(subset=["WAITTIME", "REGION", "sex_label"]).assign(
    region_name=lambda d: d["REGION"].map(REGION_NAMES)
)

region_stats = (
    valid.groupby(["region_name", "sex_label"])
    .agg(
        median_wait=("WAITTIME", "median"),
        mean_wait=("WAITTIME", "mean"),
        n=("WAITTIME", "size"),
    )
    .round(2)
)

wide = pd.DataFrame(
    {
        "female_median_wait": region_stats.xs("Female", level="sex_label")[
            "median_wait"
        ],
        "male_median_wait": region_stats.xs("Male", level="sex_label")["median_wait"],
        "female_mean_wait": region_stats.xs("Female", level="sex_label")["mean_wait"],
        "male_mean_wait": region_stats.xs("Male", level="sex_label")["mean_wait"],
        "female_n": region_stats.xs("Female", level="sex_label")["n"].astype(int),
        "male_n": region_stats.xs("Male", level="sex_label")["n"].astype(int),
    }
)
wide["gap_min"] = (wide["female_median_wait"] - wide["male_median_wait"]).round(2)
wide["total_n"] = wide["female_n"] + wide["male_n"]
wide = wide.reindex(["Northeast", "Midwest", "South", "West"])

print("Median ED wait time by region and sex (2009\u20132022):")
wide

Median ED wait time by region and sex (2009–2022):


,female_median_wait,male_median_wait,female_mean_wait,male_mean_wait,female_n,male_n,gap_min,total_n
region_name,,,,,,,,
Northeast,25.0,24.0,49.74,47.95,30218,26710,1.0,56928
Midwest,19.0,17.0,39.25,37.44,36808,30244,2.0,67052
South,23.0,22.0,47.44,45.41,54087,42598,1.0,96685
West,18.0,17.0,38.86,37.32,30017,26669,1.0,56686


## 3. Build the interactive choropleth

Plotly's `locationmode='USA-states'` colours individual states by 2-letter code. Since NHAMCS only gives us 4 regions, every state in a region gets the same value. The hover text shows region name, female median wait, male median wait, and total N.

In [4]:
STATE_TO_REGION = {
    "CT": "Northeast",
    "ME": "Northeast",
    "MA": "Northeast",
    "NH": "Northeast",
    "RI": "Northeast",
    "VT": "Northeast",
    "NJ": "Northeast",
    "NY": "Northeast",
    "PA": "Northeast",
    "IL": "Midwest",
    "IN": "Midwest",
    "MI": "Midwest",
    "OH": "Midwest",
    "WI": "Midwest",
    "IA": "Midwest",
    "KS": "Midwest",
    "MN": "Midwest",
    "MO": "Midwest",
    "NE": "Midwest",
    "ND": "Midwest",
    "SD": "Midwest",
    "DE": "South",
    "FL": "South",
    "GA": "South",
    "MD": "South",
    "NC": "South",
    "SC": "South",
    "VA": "South",
    "DC": "South",
    "WV": "South",
    "AL": "South",
    "KY": "South",
    "MS": "South",
    "TN": "South",
    "AR": "South",
    "LA": "South",
    "OK": "South",
    "TX": "South",
    "AZ": "West",
    "CO": "West",
    "ID": "West",
    "MT": "West",
    "NV": "West",
    "NM": "West",
    "UT": "West",
    "WY": "West",
    "AK": "West",
    "CA": "West",
    "HI": "West",
    "OR": "West",
    "WA": "West",
}

STATE_NAMES = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "DC": "D.C.",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",
}

states_df = pd.DataFrame(
    [(code, STATE_NAMES[code], region) for code, region in STATE_TO_REGION.items()],
    columns=["state_code", "state_name", "region"],
)
states_df = states_df.merge(wide, left_on="region", right_index=True)

states_df.head()

,state_code,state_name,region,female_median_wait,male_median_wait,female_mean_wait,male_mean_wait,female_n,male_n,gap_min,total_n
0,CT,Connecticut,Northeast,25.0,24.0,49.74,47.95,30218,26710,1.0,56928
1,ME,Maine,Northeast,25.0,24.0,49.74,47.95,30218,26710,1.0,56928
2,MA,Massachusetts,Northeast,25.0,24.0,49.74,47.95,30218,26710,1.0,56928
3,NH,New Hampshire,Northeast,25.0,24.0,49.74,47.95,30218,26710,1.0,56928
4,RI,Rhode Island,Northeast,25.0,24.0,49.74,47.95,30218,26710,1.0,56928


In [5]:
states_df["hover"] = states_df.apply(
    lambda r: (
        f"<b>{r['state_name']}</b><br>"
        f"<i>{r['region']} region</i><br><br>"
        f"\u2640 Female wait: <b>{r['female_median_wait']:.0f} min</b>"
        f" &nbsp;(n={r['female_n']:,})<br>"
        f"\u2642 Male wait: &nbsp;&nbsp;<b>{r['male_median_wait']:.0f} min</b>"
        f" &nbsp;(n={r['male_n']:,})<br>"
        f"<br>Gap (F\u2212M): <b>{r['gap_min']:+.0f} min</b><br>"
        f"Total visits: {r['total_n']:,}"
    ),
    axis=1,
)

fig = go.Figure(
    go.Choropleth(
        locations=states_df["state_code"],
        locationmode="USA-states",
        z=states_df["female_median_wait"],
        text=states_df["hover"],
        hovertemplate="%{text}<extra></extra>",
        colorscale="Reds",
        colorbar=dict(
            title=dict(text="Female<br>median<br>wait (min)", font=dict(size=12)),
            thickness=14,
            len=0.7,
        ),
        marker_line_color="white",
        marker_line_width=0.5,
        zmin=states_df["female_median_wait"].min(),
        zmax=states_df["female_median_wait"].max(),
    )
)

fig.update_layout(
    title=dict(
        text=(
            "Median ED wait time for women, by US Census region"
            "<br><sub>NHAMCS 2009\u20132022 \u00b7 darker red = women wait longer"
            " \u00b7 hover for male/female breakdown</sub>"
        ),
        x=0.5,
        xanchor="center",
    ),
    geo=dict(
        scope="usa",
        projection=dict(type="albers usa"),
        showlakes=True,
        lakecolor="rgb(245, 245, 245)",
        bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=10, r=10, t=80, b=10),
    height=550,
)

fig.show()

## 4. Companion view — the female-minus-male gap, by region

The map above colours by the absolute female wait. This second figure colours by the *gap* (Female − Male), so the regions where the disparity is widest pop out regardless of overall pace.

In [6]:
fig_gap = go.Figure(
    go.Choropleth(
        locations=states_df["state_code"],
        locationmode="USA-states",
        z=states_df["gap_min"],
        text=states_df["hover"],
        hovertemplate="%{text}<extra></extra>",
        colorscale="Reds",
        colorbar=dict(
            title=dict(text="Gap F\u2212M<br>(min)", font=dict(size=12)),
            thickness=14,
            len=0.7,
        ),
        marker_line_color="white",
        marker_line_width=0.5,
    )
)

fig_gap.update_layout(
    title=dict(
        text=(
            "Female \u2212 Male wait-time gap, by US Census region"
            "<br><sub>NHAMCS 2009\u20132022 \u00b7 darker red = wider disparity</sub>"
        ),
        x=0.5,
        xanchor="center",
    ),
    geo=dict(scope="usa", projection=dict(type="albers usa")),
    margin=dict(l=10, r=10, t=80, b=10),
    height=550,
)

fig_gap.show()

## Notes & caveats

- **State granularity is unavailable.** NHAMCS public-use files only release `REGION` (4 Census regions). Every state on the map shares its region's value — the map is a stylised regional view, not state-level data.
- **Survey weighting not applied.** Cell counts (`n`) are raw sampled visits. For nationally representative estimates, multiply by `PATWT`. The relative comparison between regions and sexes is unaffected by skipping the weight.
- **Years 2009–2022.** Longest stretch where `WAITTIME` is reliably populated alongside `IMMEDR` and `PAINSCALE`, matching the rest of the project.

## 5. Animated version — wait time by year

Same map, but each **frame = one year** (2009–2022). Use the slider to scrub through time, the play button for autoplay, and the dropdown to switch between *Female wait*, *Male wait*, and the *F−M gap*.

In [7]:
yearly = (
    valid.groupby(["YEAR", "region_name", "sex_label"])["WAITTIME"]
    .agg(median_wait="median", n="size")
    .reset_index()
)

yearly_wide = yearly.pivot_table(
    index=["YEAR", "region_name"],
    columns="sex_label",
    values=["median_wait", "n"],
).reset_index()
yearly_wide.columns = [
    "_".join(c).strip("_").lower() if isinstance(c, tuple) else c.lower()
    for c in yearly_wide.columns
]
yearly_wide = yearly_wide.rename(
    columns={
        "median_wait_female": "female_median_wait",
        "median_wait_male": "male_median_wait",
        "n_female": "female_n",
        "n_male": "male_n",
        "year": "year",
    }
)
yearly_wide["gap_min"] = (
    yearly_wide["female_median_wait"] - yearly_wide["male_median_wait"]
)
yearly_wide["female_n"] = yearly_wide["female_n"].astype(int)
yearly_wide["male_n"] = yearly_wide["male_n"].astype(int)
yearly_wide["total_n"] = yearly_wide["female_n"] + yearly_wide["male_n"]

state_region_df = pd.DataFrame(
    [(code, STATE_NAMES[code], region) for code, region in STATE_TO_REGION.items()],
    columns=["state_code", "state_name", "region_name"],
)

frame_long = state_region_df.merge(yearly_wide, on="region_name")
frame_long["hover"] = frame_long.apply(
    lambda r: (
        f"<b>{r['state_name']}</b><br>"
        f"<i>{r['region_name']} region · {int(r['year'])}</i><br><br>"
        f"♀ Female wait: <b>{r['female_median_wait']:.0f} min</b>"
        f" &nbsp;(n={r['female_n']:,})<br>"
        f"♂ Male wait: &nbsp;&nbsp;<b>{r['male_median_wait']:.0f} min</b>"
        f" &nbsp;(n={r['male_n']:,})<br><br>"
        f"Gap (F−M): <b>{r['gap_min']:+.0f} min</b><br>"
        f"Total visits: {r['total_n']:,}"
    ),
    axis=1,
)

years_sorted = sorted(frame_long["year"].unique().astype(int))
gap_abs_max = float(frame_long["gap_min"].abs().max())
zmin = -gap_abs_max
zmax = gap_abs_max


def trace_for(year: int) -> go.Choropleth:
    sub = frame_long[frame_long["year"] == year]
    return go.Choropleth(
        locations=sub["state_code"],
        locationmode="USA-states",
        z=sub["gap_min"],
        text=sub["hover"],
        hovertemplate="%{text}<extra></extra>",
        colorscale="RdBu_r",
        zmid=0,
        zmin=zmin,
        zmax=zmax,
        colorbar=dict(
            title=dict(text="Gap F−M<br>(min)", font=dict(size=12)),
            thickness=14,
            len=0.7,
        ),
        marker_line_color="white",
        marker_line_width=0.5,
    )


fig_anim = go.Figure(
    data=[trace_for(years_sorted[0])],
    frames=[go.Frame(data=[trace_for(y)], name=str(y)) for y in years_sorted],
)

play_args = [
    None,
    dict(
        frame=dict(duration=700, redraw=True),
        transition=dict(duration=300, easing="cubic-in-out"),
        fromcurrent=True,
        mode="immediate",
    ),
]
pause_args = [
    [None],
    dict(
        frame=dict(duration=0, redraw=False),
        mode="immediate",
        transition=dict(duration=0),
    ),
]

fig_anim.update_layout(
    title=dict(
        text=(
            "Female − Male ED wait-time gap, by US Census region (2009–2022)"
            "<br><sub>NHAMCS · slider scrubs years · red = women wait longer, blue = men wait longer</sub>"
        ),
        x=0.5,
        xanchor="center",
    ),
    geo=dict(
        scope="usa",
        projection=dict(type="albers usa"),
        showlakes=True,
        lakecolor="rgb(245, 245, 245)",
        bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=10, r=10, t=90, b=10),
    height=600,
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.02,
            y=-0.05,
            xanchor="left",
            yanchor="top",
            showactive=False,
            buttons=[
                dict(
                    label="▶ / ⏸",
                    method="animate",
                    args=play_args,
                    args2=pause_args,
                ),
            ],
        ),
    ],
    sliders=[
        dict(
            active=0,
            x=0.15,
            y=-0.05,
            len=0.8,
            xanchor="left",
            yanchor="top",
            currentvalue=dict(
                prefix="Year: ", visible=True, xanchor="right", font=dict(size=14)
            ),
            transition=dict(duration=300, easing="cubic-in-out"),
            pad=dict(b=10, t=10),
            steps=[
                dict(
                    label=str(y),
                    method="animate",
                    args=[
                        [str(y)],
                        dict(
                            mode="immediate",
                            frame=dict(duration=300, redraw=True),
                            transition=dict(duration=300),
                        ),
                    ],
                )
                for y in years_sorted
            ],
        )
    ],
)

fig_anim.show()